# HOL-01 — Verified Institution Research Agent

Build a **grounded, self-verifying research agent** on Snowflake — the same pattern as the production *Research Workflows* app, on **free, portable public data**.

**You will build:** given a bank + peers, produce a one-page research note (scale, profitability, capital) using *only* governed FFIEC/FDIC call-report data, then run a **verifier** that strips any unsourced claim and flags hedged reports as *Needs review*.

| Step | What you build | Production analog |
|---|---|---|
| 1 | Explore the governed data | the curated data packs |
| 2 | Query the **semantic view** (Cortex Analyst) | the Analyst tool |
| 3 | Build a verified **fact pack** | `get_fact_pack` |
| 4 | **Grounded draft** under DATA RULES | the research agent |
| 5 | **Verify** claims vs evidence | `VERIFY_REPORT` |
| 6 | Persist the note | `WORKFLOW_OUTPUTS` |

**Prerequisite:** run `setup.sql` first (creates `SNOWPACK_BANK_RESEARCH.CORE`: the fact view, the semantic view, and the `REPORTS` table). All data comes from the free `SNOWFLAKE_PUBLIC_DATA_FINANCIAL_PACKAGE` Marketplace listing.

In [ ]:
# Parameters — edit these, then Run All
SCHEMA        = "SNOWPACK_BANK_RESEARCH.CORE"
MODEL         = "claude-sonnet-4-5"   # any Cortex model your account has, e.g. llama3.1-70b, mistral-large2
SUBJECT_BANK  = "JPMorgan Chase Bank, National Association"
PEER_BANKS    = [
    "Bank of America, National Association",
    "Wells Fargo Bank, National Association",
    "Citibank, National Association",
]
AS_OF_QUARTER = "2026-06-30"   # reporting quarter to analyze

import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print("Connected as", session.sql("select current_user(), current_role()").collect()[0])

## Step 1 — Explore the governed data

`BANK_QUARTERLY_METRICS` is one clean row per (bank, quarter). It is the agent's **only** allowed source of numbers — so the agent cannot invent figures.

In [ ]:
names = "', '".join([SUBJECT_BANK] + PEER_BANKS)
df = session.sql(f"""
    SELECT bank_name, state, quarter_end,
           total_assets, total_deposits, net_income_qtr,
           return_on_assets, return_on_equity, equity_to_assets
    FROM {SCHEMA}.BANK_QUARTERLY_METRICS
    WHERE bank_name IN ('{names}') AND quarter_end = '{AS_OF_QUARTER}'
    ORDER BY total_assets DESC
""").to_pandas()
df

## Step 2 — Cortex Analyst over the semantic view

`BANK_RESEARCH_SV` adds business meaning (synonyms, metric definitions) on top of the view so **Cortex Analyst** can answer natural-language questions. Below we query it directly with the `SEMANTIC_VIEW(...)` syntax; in Snowflake Intelligence / a Cortex Agent, Analyst writes this SQL for you from a plain-English question like *"top 5 banks by total assets in 2026"*.

In [ ]:
session.sql(f"""
    SELECT * FROM SEMANTIC_VIEW({SCHEMA}.BANK_RESEARCH_SV
        METRICS total_assets_usd, avg_return_on_assets
        DIMENSIONS bank_name, report_year)
    WHERE report_year = YEAR('{AS_OF_QUARTER}'::date)
    ORDER BY total_assets_usd DESC
    LIMIT 10
""").to_pandas()

## Step 3 — Build the verified fact pack

The fact pack is a small JSON bundle of **only** the verified rows the agent may cite — the subject bank and peers, latest quarter plus one year prior (for a trend). This is exactly how `get_fact_pack` works in production: precomputed, governed, copy-exactly.

In [ ]:
fact_pack = session.sql(f"""
    SELECT ARRAY_AGG(OBJECT_CONSTRUCT(
             'bank', bank_name, 'quarter_end', quarter_end::string,
             'total_assets_usd', total_assets, 'total_deposits_usd', total_deposits,
             'net_income_qtr_usd', net_income_qtr,
             'roa_fraction', return_on_assets, 'roe_fraction', return_on_equity,
             'equity_to_assets_fraction', equity_to_assets,
             'ci_loans_usd', ci_loans, 'credit_card_loans_usd', credit_card_loans)) AS pack
    FROM {SCHEMA}.BANK_QUARTERLY_METRICS
    WHERE bank_name IN ('{names}')
      AND quarter_end IN ('{AS_OF_QUARTER}', DATEADD(year, -1, '{AS_OF_QUARTER}')::date)
""").collect()[0][0]
print(fact_pack[:800], '...')

## Step 4 — Grounded draft (DATA RULES)

The system prompt is the heart of a *verified* agent. The **DATA RULES** forbid outside numbers, estimates, and hedging — the model may only use the fact pack. This mirrors the production `WORKFLOW_TEMPLATES.prompt_template`.

In [ ]:
DATA_RULES = (
    "You are a buy-side bank analyst. Write a concise peer research note on the SUBJECT bank versus its peers "
    "for the reporting quarter, using ONLY the FACTS JSON below.\n\n"
    "DATA RULES (mandatory):\n"
    "1. Use ONLY numbers present in FACTS. Copy them exactly. Never estimate, extrapolate, or add outside figures.\n"
    "2. roa_fraction / roe_fraction / equity_to_assets_fraction are fractions (0.015 = 1.5%). Convert to % when writing.\n"
    "3. Dollar fields are already in USD. Express in $B/$T.\n"
    "4. No hedging words (likely, approximately, around, could, industry-leading) and no consensus/market claims.\n"
    "5. If a figure is missing, write n/d. Do not invent a row.\n"
    "OUTPUT: markdown starting with '# ', a one-line takeaway, a peer comparison table, then 3 short bullets. "
    "End with a '## Data Coverage' line naming the source (FFIEC/FDIC call reports) and the quarter(s) used."
)
prompt = DATA_RULES + f"\n\nSUBJECT: {SUBJECT_BANK}\nREPORTING QUARTER: {AS_OF_QUARTER}\nFACTS: {fact_pack}"
draft_md = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, prompt]).collect()[0][0]
print(draft_md)

## Step 5 — Verify the draft against the evidence

A second model pass fact-checks every claim against the fact pack and counts residual hedges. If anything is unsupported or hedged, the note is flagged **Needs review** — the same quality gate as the production `VERIFY_REPORT`.

In [ ]:
verify_prompt = (
    "You are a fact-checker. Compare the DRAFT against the FACTS JSON (the only allowed source). "
    "Return ONLY a valid JSON object (double quotes on all keys and string values, no code fences, no prose) "
    "with keys: status (\"verified\" if every number and claim is supported by FACTS and there are no hedge "
    "words like likely/approximately/around/could/industry-leading, else \"needs_review\"), "
    "residual_hedges (integer), edits (array of short strings naming each unsupported or hedged claim).\n\n"
    f"FACTS: {fact_pack}\n\nDRAFT:\n{draft_md}"
)
raw = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, verify_prompt]).collect()[0][0]
# models sometimes wrap JSON in prose/fences — extract the object and parse defensively
import re as _re
def _parse_verdict(x):
    t = x.strip()
    for _ in range(3):
        m = _re.match(r"^```[a-zA-Z]*\s*(.*?)\s*```$", t, _re.S)
        if m:
            t = m.group(1).strip()
        try:
            v = json.loads(t)
        except Exception:
            a, b = t.find("{"), t.rfind("}")
            if a >= 0 and b > a:
                try:
                    v = json.loads(t[a:b + 1])
                except Exception:
                    return None
            else:
                return None
        if isinstance(v, str):
            t = v.strip()
            continue
        return v
    return None
verdict = _parse_verdict(raw) or {"status": "needs_review", "residual_hedges": None, "edits": ["unparseable verifier output"]}  # keep the safe fallback
status = verdict.get("status", "needs_review")
edits = verdict.get("edits", []) or []
hedges = verdict.get("residual_hedges")
print(f"STATUS: {status.upper()}   residual_hedges={hedges}   edits={len(edits)}")
for e in edits:
    print(" -", e)

## Step 6 — Persist the note

Save the verified note to `REPORTS`, tagged with its status — the store a Streamlit gallery or a scheduled task would read from.

In [ ]:
session.sql(
    f"INSERT INTO {SCHEMA}.REPORTS (subject, draft_md, report_md, status, edits_applied, residual_hedges) "
    f"SELECT ?, ?, ?, ?, ?, ?",
    params=[SUBJECT_BANK, draft_md, draft_md, status, len(edits), hedges]
).collect()
session.sql(f"SELECT run_id, subject, status, edits_applied, residual_hedges, created_at FROM {SCHEMA}.REPORTS ORDER BY created_at DESC LIMIT 5").to_pandas()

## Extension A — Turn it into a real Cortex Agent

Steps 3–5 showed the *verified* pattern explicitly. To give a PM a conversational interface, wrap the semantic view in a **Cortex Agent** with a Cortex Analyst tool and the same grounding instructions. Run the SQL below in a SQL cell (or Snowsight worksheet):

```sql
CREATE OR REPLACE AGENT SNOWPACK_BANK_RESEARCH.CORE.BANK_RESEARCH_AGENT
  COMMENT = 'HOL-01 verified bank research agent (Cortex Analyst over BANK_RESEARCH_SV)'
  PROFILE = '{"display_name": "Bank Research Agent", "color": "blue"}'
  FROM SPECIFICATION
  $$
  models:
    orchestration: auto
  orchestration:
    capabilities:
      analytical_search: true
  instructions:
    response: 'You are a buy-side bank analyst. Use ONLY figures returned by the Analyst tool; never invent numbers. ROA/ROE/equity ratios are fractions (0.015 = 1.5%). Express dollars in $B/$T. Be concise and cite the quarter.'
    orchestration: 'For any bank metric, ranking, or comparison, use the BankMetrics Analyst tool.'
    sample_questions:
      - question: 'Top 5 banks by total assets in 2026'
      - question: "What was JPMorgan Chase Bank's ROA and ROE in Q2 2026?"
  tools:
    - tool_spec:
        type: cortex_analyst_text_to_sql
        name: BankMetrics
        description: 'Quarterly FFIEC/FDIC call-report metrics (assets, deposits, net income, ROA, ROE, capital, loans) per US bank.'
  tool_resources:
    BankMetrics:
      semantic_view: SNOWPACK_BANK_RESEARCH.CORE.BANK_RESEARCH_SV
  $$;
```

**Chat with it:** open Snowsight → **AI & ML → Agents → Bank Research Agent** (or add it to CoWork). The agent runs with the role selected in the UI, so that role needs `USAGE` on the agent and on the semantic view's data. To let another role use it:

```sql
GRANT USAGE ON AGENT SNOWPACK_BANK_RESEARCH.CORE.BANK_RESEARCH_AGENT TO ROLE <role>;
```

> The agent reuses the **same** semantic view you built, so Cortex Analyst only returns governed numbers — the grounding guarantee carries over to the conversational experience.

## Recap & extensions

You built the full **verified research agent** loop: governed semantic view → verified fact pack → grounded draft under DATA RULES → verification gate → stored note. Everything ran on free, portable public data.

**Graduation extensions (try next):**
1. **Turn it into a real Cortex Agent** — done in *Extension A* above (`CREATE AGENT` over `BANK_RESEARCH_SV`); chat with it in Snowflake Intelligence / CoWork.
2. **Add an unstructured tool** — build a `CORTEX SEARCH SERVICE` over `FINANCIAL_CFPB_COMPLAINT` narratives (scope it tightly) and let the agent cite consumer-complaint themes.
3. **Schedule it** — wrap Steps 3–6 in a stored proc and a `TASK` to run a watchlist weekly (like `WEEKLY_RESEARCH_WORKFLOWS`).
4. **Surface it** — a Streamlit card gallery reading `REPORTS`, mirroring the production Research Workflows app.

**Try breaking the verifier:** set `MODEL` to a weaker model, or edit the draft to add a made-up number, and watch Step 5 flag it as *Needs review*.